In [ ]:
def mu_nmf(V, k, max_iter=400, tolerance=1e-4):
    m, n = V.shape
    W = np.random.rand(m, k) #Random ma trận W
    H = np.random.rand(k, n)  #Random ma trận H
    last_error = np.inf   #Lỗi = oo
    for i in range(max_iter):
        # Cập nhật H
        numerator_H = W.T @ V
        denominator_H = (W.T @ W) @ H
        denominator_H[denominator_H == 0] = 1e-9
        H = H * (numerator_H / denominator_H)

        # Cập nhật W
        numerator_W = V @ H.T
        denominator_W = W @ (H @ H.T)
        denominator_W[denominator_W == 0] = 1e-9
        W = W * (numerator_W / denominator_W)

        if (i + 1) % 10 == 0:
            error = np.linalg.norm(V - W @ H, 'fro')
            if np.abs(last_error - error) < tolerance:
                break #Lặp nhiều mà lỗi ko giảm => break sớm
            last_error = error

    print(f"Hoàn thành NMF sau {i+1} vòng lặp với lỗi cuối cùng = {last_error:.4f}")
    return W, H

# --- BƯỚC 4: ÁP DỤNG NMF ---

# 1. Chọn số chủ đề muốn tìm
num_topics = 3

# 2. Chuyển vị ma trận V và chạy thuật toán
print(f"Kích thước V ban đầu: {V.shape}")
print(f"Kích thước V sau chuyển vị (V.T): {V.T.shape}\n")

W, H = mu_nmf(V.T, k=num_topics)

# 3. Kiểm tra kết quả
print("\nĐã phân rã ma trận V thành công!")
print(f"Kích thước ma trận W (Từ - Chủ đề): {W.shape}")
print(f"Kích thước ma trận H (Chủ đề - Văn bản): {H.shape}")

Kích thước V ban đầu: (9, 50)
Kích thước V sau chuyển vị (V.T): (50, 9)

Hoàn thành NMF sau 30 vòng lặp với lỗi cuối cùng = 2.4495

Đã phân rã ma trận V thành công!
Kích thước ma trận W (Từ - Chủ đề): (50, 3)
Kích thước ma trận H (Chủ đề - Văn bản): (3, 9)


In [ ]:
def mu_nmf(V, k, max_iter=400, tolerance=1e-4):
    m, n = V.shape
    W = np.random.rand(m, k) #Random ma trận W
    H = np.random.rand(k, n)  #Random ma trận H
    last_error = np.inf   #Lỗi = oo
    for i in range(max_iter):
        # Cập nhật H
        numerator_H = W.T @ V
        denominator_H = (W.T @ W) @ H
        denominator_H[denominator_H == 0] = 1e-9
        H = H * (numerator_H / denominator_H)

        # Cập nhật W
        numerator_W = V @ H.T
        denominator_W = W @ (H @ H.T)
        denominator_W[denominator_W == 0] = 1e-9
        W = W * (numerator_W / denominator_W)

        if (i + 1) % 10 == 0:
            error = np.linalg.norm(V - W @ H, 'fro')
            if np.abs(last_error - error) < tolerance:
                break #Lặp nhiều mà lỗi ko giảm => break sớm
            last_error = error

    print(f"Hoàn thành NMF sau {i+1} vòng lặp với lỗi cuối cùng = {last_error:.4f}")
    return W, H

# --- BƯỚC 4: ÁP DỤNG NMF ---

# 1. Chọn số chủ đề muốn tìm
num_topics = 3

# 2. Chuyển vị ma trận V và chạy thuật toán
print(f"Kích thước V ban đầu: {V.shape}")
print(f"Kích thước V sau chuyển vị (V.T): {V.T.shape}\n")

W, H = mu_nmf(V.T, k=num_topics)

# 3. Kiểm tra kết quả
print("\nĐã phân rã ma trận V thành công!")
print(f"Kích thước ma trận W (Từ - Chủ đề): {W.shape}")
print(f"Kích thước ma trận H (Chủ đề - Văn bản): {H.shape}")

Kích thước V ban đầu: (9, 50)
Kích thước V sau chuyển vị (V.T): (50, 9)

Hoàn thành NMF sau 30 vòng lặp với lỗi cuối cùng = 2.4495

Đã phân rã ma trận V thành công!
Kích thước ma trận W (Từ - Chủ đề): (50, 3)
Kích thước ma trận H (Chủ đề - Văn bản): (3, 9)


In [ ]:
!pip install pyvi

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import math
import requests
from collections import Counter
import re
from pyvi import ViTokenizer
import os

1. Nạp data

In [ ]:
documents = [
    "Giá vàng hôm nay tăng mạnh, nhà đầu tư rất phấn khởi.",
    "Thị trường chứng khoán biến động, chỉ số VN-Index giảm điểm.",
    "Lãi suất ngân hàng được dự báo sẽ ổn định trong quý tới.",
    "Cập nhật công nghệ AI mới nhất, giúp tự động hóa nhiều quy trình.",
    "Ra mắt điện thoại thông minh với chip xử lý mạnh mẽ và camera siêu nét.",
    "Trí tuệ nhân tạo đang thay đổi cách chúng ta làm việc và sống.",
    "Đội tuyển bóng đá quốc gia chuẩn bị cho trận đấu quan trọng.",
    "Giải vô địch bóng chuyền nữ thu hút đông đảo người hâm mộ.",
    "Lịch thi đấu World Cup đã được công bố chính thức."
]

2. Tiền xử lý data

In [ ]:
# Xây dựng từ điển các từ dừng
stopwords = "https://raw.githubusercontent.com/stopwords/vietnamese-stopwords/refs/heads/master/vietnamese-stopwords.txt"

response = requests.get(stopwords)
vietnamese_stopwords = response.text.splitlines()

def preprocess_text(text):
    # Chuyển về chữ thường
    text = text.lower()
    # Loại bỏ chữ số, ký tự đặc biệt
    text = re.sub(r'[^a-zàáạảãăắằặẳẵâấầậẩẫèéẹẻẽêếềệểễìíịỉĩòóọỏõôốồộổỗơớờợởỡùúụủũưứừựửữýỳỵỷỹđ\s]', '', text)
    # Tách từ
    tokens = ViTokenizer.tokenize(text).split()
    processed_tokens = [
        token
        for token in tokens
        if token.replace('_', ' ') not in vietnamese_stopwords and len(token) > 1
    ]
    # Loại bỏ từ dừng
    processed_tokens = [token for token in processed_tokens if token not in vietnamese_stopwords]
    return processed_tokens

preprocessed_documents = [preprocess_text(doc) for doc in documents]

for doc in preprocessed_documents:
    print(doc)

['giá', 'vàng', 'hôm_nay', 'đầu_tư', 'phấn_khởi']
['thị_trường', 'chứng_khoán', 'biến_động', 'chỉ_số', 'vnindex']
['lãi_suất', 'ngân_hàng', 'dự_báo', 'ổn_định', 'quý']
['cập_nhật', 'công_nghệ', 'giúp', 'tự_động', 'hóa', 'quy_trình']
['ra_mắt', 'điện_thoại', 'thông_minh', 'chip', 'mạnh_mẽ', 'camera', 'siêu', 'nét']
['trí_tuệ', 'nhân_tạo', 'làm_việc', 'sống']
['đội_tuyển', 'bóng_đá', 'quốc_gia', 'trận', 'đấu']
['giải', 'vô_địch', 'bóng_chuyền', 'nữ', 'thu_hút', 'đông_đảo', 'hâm_mộ']
['lịch', 'thi_đấu', 'world_cup', 'công_bố', 'chính_thức']


3. Vector hóa, xây dựng ma trận V

In [ ]:
joined_preprocessed_documents = [' '.join(doc) for doc in preprocessed_documents]

vectorizer = TfidfVectorizer(smooth_idf=True, use_idf=True)
V = vectorizer.fit_transform(joined_preprocessed_documents).toarray()

feature_names = vectorizer.get_feature_names_out()

print("Kích thước ma trận V:", V.shape)

df_V = pd.DataFrame(V, columns=feature_names)
print("\nMa trận V (TF-IDF):")
print(df_V)

Kích thước ma trận V: (9, 50)

Ma trận V (TF-IDF):
   biến_động  bóng_chuyền   bóng_đá    camera      chip  chính_thức    chỉ_số  \
0   0.000000     0.000000  0.000000  0.000000  0.000000    0.000000  0.000000   
1   0.447214     0.000000  0.000000  0.000000  0.000000    0.000000  0.447214   
2   0.000000     0.000000  0.000000  0.000000  0.000000    0.000000  0.000000   
3   0.000000     0.000000  0.000000  0.000000  0.000000    0.000000  0.000000   
4   0.000000     0.000000  0.000000  0.353553  0.353553    0.000000  0.000000   
5   0.000000     0.000000  0.000000  0.000000  0.000000    0.000000  0.000000   
6   0.000000     0.000000  0.447214  0.000000  0.000000    0.000000  0.000000   
7   0.000000     0.377964  0.000000  0.000000  0.000000    0.000000  0.000000   
8   0.000000     0.000000  0.000000  0.000000  0.000000    0.447214  0.000000   

   chứng_khoán   công_bố  công_nghệ  ...   vnindex      vàng   vô_địch  \
0     0.000000  0.000000   0.000000  ...  0.000000  0.447214  0.

4. Sử dụng NMF - MU

In [ ]:
def mu_nmf(V, k, max_iter=400, tolerance=1e-4):
    m, n = V.shape
    W = np.random.rand(m, k) #Random ma trận W
    H = np.random.rand(k, n)  #Random ma trận H
    last_error = np.inf   #Lỗi = oo
    for i in range(max_iter):
        # Cập nhật H
        numerator_H = W.T @ V
        denominator_H = (W.T @ W) @ H
        denominator_H[denominator_H == 0] = 1e-9
        H = H * (numerator_H / denominator_H)

        # Cập nhật W
        numerator_W = V @ H.T
        denominator_W = W @ (H @ H.T)
        denominator_W[denominator_W == 0] = 1e-9
        W = W * (numerator_W / denominator_W)

        if (i + 1) % 10 == 0:
            error = np.linalg.norm(V - W @ H, 'fro')
            if np.abs(last_error - error) < tolerance:
                break #Lặp nhiều mà lỗi ko giảm => break sớm
            last_error = error

    print(f"Hoàn thành NMF sau {i+1} vòng lặp với lỗi cuối cùng = {last_error:.4f}")
    return W, H

# --- BƯỚC 4: ÁP DỤNG NMF ---

# 1. Chọn số chủ đề muốn tìm
num_topics = 3

# 2. Chuyển vị ma trận V và chạy thuật toán
print(f"Kích thước V ban đầu: {V.shape}")
print(f"Kích thước V sau chuyển vị (V.T): {V.T.shape}\n")

W, H = mu_nmf(V.T, k=num_topics)

# 3. Kiểm tra kết quả
print("\nĐã phân rã ma trận V thành công!")
print(f"Kích thước ma trận W (Từ - Chủ đề): {W.shape}")
print(f"Kích thước ma trận H (Chủ đề - Văn bản): {H.shape}")

Kích thước V ban đầu: (9, 50)
Kích thước V sau chuyển vị (V.T): (50, 9)

Hoàn thành NMF sau 30 vòng lặp với lỗi cuối cùng = 2.4495

Đã phân rã ma trận V thành công!
Kích thước ma trận W (Từ - Chủ đề): (50, 3)
Kích thước ma trận H (Chủ đề - Văn bản): (3, 9)


In [ ]:
# --- DIỄN GIẢI MA TRẬN W ---
# Số lượng từ khóa hàng đầu muốn hiển thị cho mỗi chủ đề
num_top_words = 6

print("--- CÁC CHỦ ĐỀ ĐƯỢC KHÁM PHÁ ---")
# Lặp qua từng chủ đề (từng cột của W)
for topic_idx in range(W.shape[1]):
    print(f"\n## Chủ đề #{topic_idx + 1}:")

    # Lấy cột tương ứng với chủ đề
    topic_column = W[:, topic_idx]

    # Sắp xếp các chỉ số của từ theo điểm số giảm dần
    # argsort() trả về chỉ số, [::-1] để đảo ngược thành thứ tự giảm dần
    top_word_indices = topic_column.argsort()[::-1]

    # In ra các từ khóa hàng đầu
    top_words = [sorted_vocab[i] for i in top_word_indices[:num_top_words]]
    print("   -> Từ khóa:", ", ".join(top_words))

--- CÁC CHỦ ĐỀ ĐƯỢC KHÁM PHÁ ---

## Chủ đề #1:
   -> Từ khóa: thi đấu, world cup, công bố, chính thức, lịch, thị trường

## Chủ đề #2:
   -> Từ khóa: ổn định, quý, dự báo, lãi suất, ngân hàng, ra mắt

## Chủ đề #3:
   -> Từ khóa: trí tuệ, sống, nhân tạo, làm việc, đầu tư, vàng


In [ ]:
# --- DIỄN GIẢI MA TRẬN H ---
# Gán nhãn cho các chủ đề vừa tìm được
topic_labels_mu = {
    0: "Công nghệ",
    1: "Tài chính - Đầu tư",
    2: "Thể thao"
}

print("\n--- PHÂN LOẠI VĂN BẢN VÀO CHỦ ĐỀ ---")
# Lặp qua từng văn bản (từng cột của H)
for doc_idx in range(H.shape[1]):
    # Lấy cột tương ứng với văn bản
    doc_column = H[:, doc_idx]

    # Tìm chỉ số của chủ đề có điểm số cao nhất
    dominant_topic_idx = doc_column.argmax()

    print(f"\n## Văn bản #{doc_idx + 1}: '{documents[doc_idx]}'")
    print(f"   -> Chủ đề chính: {topic_labels_mu.get(dominant_topic_idx, 'Chưa xác định')}")


--- PHÂN LOẠI VĂN BẢN VÀO CHỦ ĐỀ ---

## Văn bản #1: 'Giá vàng hôm nay tăng mạnh, nhà đầu tư rất phấn khởi.'
   -> Chủ đề chính: Thể thao

## Văn bản #2: 'Thị trường chứng khoán biến động, chỉ số VN-Index giảm điểm.'
   -> Chủ đề chính: Công nghệ

## Văn bản #3: 'Lãi suất ngân hàng được dự báo sẽ ổn định trong quý tới.'
   -> Chủ đề chính: Tài chính - Đầu tư

## Văn bản #4: 'Cập nhật công nghệ AI mới nhất, giúp tự động hóa nhiều quy trình.'
   -> Chủ đề chính: Công nghệ

## Văn bản #5: 'Ra mắt điện thoại thông minh với chip xử lý mạnh mẽ và camera siêu nét.'
   -> Chủ đề chính: Tài chính - Đầu tư

## Văn bản #6: 'Trí tuệ nhân tạo đang thay đổi cách chúng ta làm việc và sống.'
   -> Chủ đề chính: Thể thao

## Văn bản #7: 'Đội tuyển bóng đá quốc gia chuẩn bị cho trận đấu quan trọng.'
   -> Chủ đề chính: Công nghệ

## Văn bản #8: 'Giải vô địch bóng chuyền nữ thu hút đông đảo người hâm mộ.'
   -> Chủ đề chính: Thể thao

## Văn bản #9: 'Lịch thi đấu World Cup đã được công bố chính thức.'
 

5. Sử dụng NMF - ALS

In [ ]:
def als_nmf(V, k, max_iter=400, tolerance=1e-4):
    m, n = V.shape
    # Khởi tạo W và H với giá trị ngẫu nhiên không âm
    W = np.random.rand(m, k)
    H = np.random.rand(k, n)

    last_error = np.inf

    for i in range(max_iter):
        # Cố định W, cập nhật H
        H = np.linalg.pinv(W.T @ W) @ W.T @ V
        # Ép các giá trị âm về 0 để đảm bảo tính không âm
        H[H < 0] = 0

        # Cố định H, cập nhật W
        W = V @ H.T @ np.linalg.pinv(H @ H.T)
        # Ép các giá trị âm về 0 để đảm bảo tính không âm
        W[W < 0] = 0

        # Kiểm tra điều kiện dừng sớm
        if (i + 1) % 10 == 0:
            error = np.linalg.norm(V - W @ H, 'fro')
            if np.abs(last_error - error) < tolerance:
                print(f"Hội tụ sớm sau {i+1} vòng lặp.")
                break
            last_error = error

    print(f"Hoàn thành NMF (ALS) sau {i+1} vòng lặp với lỗi cuối cùng = {last_error:.4f}")
    return W, H

In [ ]:
# --- ÁP DỤNG NMF-ALS ---
num_topics = 3
print("--- CHẠY NMF VỚI PHƯƠNG PHÁP ALS ---")

# Gọi hàm als_nmf trên ma trận V.T của bạn
W_als, H_als = als_nmf(V.T, k=num_topics)

# --- DIỄN GIẢI KẾT QUẢ TỪ ALS ---

# 1. DIỄN GIẢI MA TRẬN W (TÌM TỪ KHÓA CHO CHỦ ĐỀ)
print("\n--- CÁC CHỦ ĐỀ ĐƯỢC KHÁM PHÁ (ALS) ---")
num_top_words = 6
topic_labels_als = {} # Chuẩn bị gán nhãn

for topic_idx in range(W_als.shape[1]):
    print(f"\n## Chủ đề #{topic_idx + 1}:")
    topic_column = W_als[:, topic_idx]
    top_word_indices = topic_column.argsort()[::-1]
    top_words = [sorted_vocab[i] for i in top_word_indices[:num_top_words]]
    print("   -> Từ khóa:", ", ".join(top_words))

--- CHẠY NMF VỚI PHƯƠNG PHÁP ALS ---
Hội tụ sớm sau 30 vòng lặp.
Hoàn thành NMF (ALS) sau 30 vòng lặp với lỗi cuối cùng = 2.4495

--- CÁC CHỦ ĐỀ ĐƯỢC KHÁM PHÁ (ALS) ---

## Chủ đề #1:
   -> Từ khóa: ổn định, quý, dự báo, lãi suất, ngân hàng, nhân tạo

## Chủ đề #2:
   -> Từ khóa: vàng, đầu tư, hôm nay, giá, phấn khởi, điện thoại

## Chủ đề #3:
   -> Từ khóa: tự động, quy trình, giúp, cập nhật, hóa, công nghệ


In [ ]:
topic_labels_als = {
    0: "Công nghệ",
    1: "Tài chính - Đầu tư",
    2: "Thể thao"
}

# 2. DIỄN GIẢI MA TRẬN H (PHÂN LOẠI VĂN BẢN)
print("\n--- PHÂN LOẠI VĂN BẢN VÀO CHỦ ĐỀ (ALS) ---")
for doc_idx in range(H_als.shape[1]):
    doc_column = H_als[:, doc_idx]
    dominant_topic_idx = doc_column.argmax()
    print(f"\n## Văn bản #{doc_idx + 1}: '{documents[doc_idx]}'")
    print(f"   -> Chủ đề chính: **{topic_labels_als.get(dominant_topic_idx, 'Chưa xác định')}**")


--- PHÂN LOẠI VĂN BẢN VÀO CHỦ ĐỀ (ALS) ---

## Văn bản #1: 'Giá vàng hôm nay tăng mạnh, nhà đầu tư rất phấn khởi.'
   -> Chủ đề chính: **Thể thao**

## Văn bản #2: 'Thị trường chứng khoán biến động, chỉ số VN-Index giảm điểm.'
   -> Chủ đề chính: **Tài chính - Đầu tư**

## Văn bản #3: 'Lãi suất ngân hàng được dự báo sẽ ổn định trong quý tới.'
   -> Chủ đề chính: **Thể thao**

## Văn bản #4: 'Cập nhật công nghệ AI mới nhất, giúp tự động hóa nhiều quy trình.'
   -> Chủ đề chính: **Công nghệ**

## Văn bản #5: 'Ra mắt điện thoại thông minh với chip xử lý mạnh mẽ và camera siêu nét.'
   -> Chủ đề chính: **Tài chính - Đầu tư**

## Văn bản #6: 'Trí tuệ nhân tạo đang thay đổi cách chúng ta làm việc và sống.'
   -> Chủ đề chính: **Tài chính - Đầu tư**

## Văn bản #7: 'Đội tuyển bóng đá quốc gia chuẩn bị cho trận đấu quan trọng.'
   -> Chủ đề chính: **Thể thao**

## Văn bản #8: 'Giải vô địch bóng chuyền nữ thu hút đông đảo người hâm mộ.'
   -> Chủ đề chính: **Công nghệ**

## Văn bản #9: 'Lịch t